<a href="https://colab.research.google.com/github/LeninGF/IAG-2024B-GenerativeQA/blob/fix-metric/question-answering-Bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question Answering Generative

- Coder: Lenin G. Falconí
- Date: 2025-01-27

https://huggingface.co/docs/transformers/en/tasks/question_answering

## Instalación de Librerías

In [1]:
!pip install datasets

In [2]:
# For Colab
!pip install python-dotenv
!pip install huggingface_hub
!pip install datasets

## Login en Huggingfaces

Se realiza el login usando archivo .env

In [3]:
# For Colab
from dotenv import load_dotenv
import os
dotenv_path = '/content/.env'
load_dotenv(dotenv_path)

True

In [4]:
import os
from huggingface_hub import login
token = os.getenv('HUGGINGFACE_TOKEN')
login(token)

In [5]:
# from huggingface_hub import notebook_login
# notebook_login()

## Carga del Dataset

Se procede a realizar la carga del dataset desde huggingface

In [6]:
from datasets import load_dataset
path2dataset = "LeninGF/robos-question-answering"
squad = load_dataset(path2dataset)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
squad

DatasetDict({
    train: Dataset({
        features: ['index', 'context', 'question', 'answer_start', 'answer_end', 'impossible_find_answer', 'answer_text', 'is_impossible', 'context_id', 'answer_text_number_words'],
        num_rows: 4572
    })
})

In [8]:
# Split the dataset (adjust test_size=0.2 as needed)
squad = squad["train"].train_test_split(test_size=0.2, seed=42)

In [9]:
squad

DatasetDict({
    train: Dataset({
        features: ['index', 'context', 'question', 'answer_start', 'answer_end', 'impossible_find_answer', 'answer_text', 'is_impossible', 'context_id', 'answer_text_number_words'],
        num_rows: 3657
    })
    test: Dataset({
        features: ['index', 'context', 'question', 'answer_start', 'answer_end', 'impossible_find_answer', 'answer_text', 'is_impossible', 'context_id', 'answer_text_number_words'],
        num_rows: 915
    })
})

In [10]:
squad["train"][0]

{'index': 466,
 'context': 'señor fiscal el dia ayer 20 de junio del 2018 a las 17h00 aproximadamente por el redondel de jaramijo donde esta la fabrica puerto mar del canton jaramijo yo iba caminando y hablando por telefono de repetente un sujeto que iva caminando me arrancho el telefono de maneta violenta y lego salio corriendo con mi telefono celular era una persona joven alta moreno y de contextura delgada solocito se pida al ecu 911 si existen camara en el lugar a fin de identificar a la persona que me robo anexo a mi denuncia factura del telefono donde consta sus caracteristicas',
 'question': '¿En qué fecha ocurrió el incidente?',
 'answer_start': 25,
 'answer_end': 45,
 'impossible_find_answer': False,
 'answer_text': '20 de junio del 2018',
 'is_impossible': '0',
 'context_id': 'context_93',
 'answer_text_number_words': 5}

In [11]:
from transformers import AutoTokenizer
# model = "distilbert/distilbert-base-uncased"
model = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model)

## Cargando Modelo Pre-Entrenado

Se considera utilizar el modelo `dccuchile/bert-base-spanish-wwm-cased` que sería un ajuste al Español

In [12]:
from transformers import AutoModelForQuestionAnswering
model = AutoModelForQuestionAnswering.from_pretrained(model)

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Preprocesamiento

In [13]:
def preprocess_function(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        truncation=True,
        padding="max_length",
        max_length=384,  # Adjust if needed
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        context = examples["context"][sample_idx]
        answer_start = examples["answer_start"][sample_idx]
        answer_end = examples["answer_end"][sample_idx]
        answer_text = examples["answer_text"][sample_idx]

        # Handle impossible answers (if your dataset has them)
        if examples["impossible_find_answer"][sample_idx] or answer_start == -1:  # answer_start = 0 hay que corregir en la generacion del dataset
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Logic to find token positions (same as before)
            sequence_ids = inputs.sequence_ids(i)
            context_start = 0
            while sequence_ids[context_start] != 1:
                context_start += 1
            context_end = len(sequence_ids) - 1
            while sequence_ids[context_end] != 1:
                context_end -= 1

            if (answer_start < offsets[context_start][0] or
                answer_end > offsets[context_end][1]):
                start_positions.append(0)
                end_positions.append(0)
            else:
                # Find token positions for answer
                start_token = context_start
                while start_token < context_end and offsets[start_token][0] <= answer_start:
                    start_token += 1
                end_token = context_end
                while end_token >= context_start and offsets[end_token][1] >= answer_end:
                    end_token -= 1
                start_positions.append(start_token - 1)
                end_positions.append(end_token + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Apply preprocessing
tokenized_dataset = squad.map(
    preprocess_function,
    batched=True,
    remove_columns=squad["train"].column_names  # Remove unused columns
)

Map:   0%|          | 0/3657 [00:00<?, ? examples/s]

Map:   0%|          | 0/915 [00:00<?, ? examples/s]

In [14]:
# from transformers import DefaultDataCollator

# data_collator = DefaultDataCollator()

In [15]:
# from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer

# model = AutoModelForQuestionAnswering.from_pretrained("distilbert/distilbert-base-uncased")

## Evaluación del Rendimiento del Modelo
Se utilizara `squad_metric` para calcular EM y F1

In [16]:
!pip install evaluate # for colab

In [17]:
import evaluate
from tqdm.auto import tqdm

qa_metric = evaluate.load("squad")

def compute_metrics(start_logits, end_logits, tokenized_dataset, original_dataset):
    # Map predicted token positions to text spans
    predicted_answers = []
    for i in tqdm(range(len(tokenized_dataset))):
        start_logit = start_logits[i]
        end_logit = end_logits[i]
        offsets = tokenized_dataset[i]["offset_mapping"]
        example_id = tokenized_dataset[i]["example_ids"]

        # Get the original example
        original_example = original_dataset[example_id]

        # Find the predicted token positions
        start_index = np.argmax(start_logit)
        end_index = np.argmax(end_logit)

        # Convert tokens to character positions
        start_char = offsets[start_index][0]
        end_char = offsets[end_index][1]

        # Extract predicted answer text
        predicted_text = original_example["context"][start_char:end_char]

        # Handle impossible answers (if applicable) Actualizar a la etiqueta correcta
        if original_example["is_impossible"]:
            predicted_text = ""

        predicted_answers.append({
            "id": str(example_id),
            "prediction_text": predicted_text,
            "no_answer_probability": 0.0 if predicted_text else 1.0,
        })

    # Format ground truth answers
    true_answers = [
        {"id": str(i), "answers": {"text": [ex["answer_text"]], "answer_start": [ex["answer_start"]]}}
        for i, ex in enumerate(original_dataset)
    ]

    # Compute metrics
    results = qa_metric.compute(predictions=predicted_answers, references=true_answers)
    return results

Para evaluar durante el entrenamiento se defina la clase `QATrainer`

In [18]:
from transformers import Trainer

class QATrainer(Trainer):
    def compute_metrics(self, eval_preds):
        start_logits, end_logits = eval_preds.predictions
        return compute_metrics(start_logits, end_logits, self.eval_dataset, dataset["test"])

## Entrenamiento

In [19]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [20]:
# os.mkdir("./models")

In [21]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./fge-robos-qa-model", # colocar dentro de models
    evaluation_strategy="epoch",
    learning_rate=2e-5,  # Slightly lower for non-English models
    per_device_train_batch_size=16,  # Adjust based on GPU memory
    per_device_eval_batch_size=16,
    num_train_epochs=10,  # Spanish datasets may need more epochs
    weight_decay=0.01,
    save_strategy="epoch",
    fp16=True,  # Use if GPU supports it
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [24]:
from transformers import Trainer

trainer = QATrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],  # Now using the test split
    tokenizer=tokenizer,
    # compute_metrics=compute_metrics
)

<ipython-input-24-3870c58528a4>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `QATrainer.__init__`. Use `processing_class` instead.
  trainer = QATrainer(


In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,1.038180
2,No log,1.059036
3,0.642100,1.170063
4,0.642100,1.393646
5,0.406600,1.541096
6,0.406600,1.597787
7,0.239800,1.746270
8,0.239800,1.814177
9,0.156100,1.865808
10,0.156100,1.895638


TrainOutput(global_step=2290, training_loss=0.33085736936877386, metrics={'train_runtime': 907.7607, 'train_samples_per_second': 40.286, 'train_steps_per_second': 2.523, 'total_flos': 7166716795376640.0, 'train_loss': 0.33085736936877386, 'epoch': 10.0})

## Evaluación


In [26]:
# Evaluate on test set
results = trainer.evaluate()
print("Test set results:", results)

Test set results: {'eval_loss': 1.895638108253479, 'eval_runtime': 5.3623, 'eval_samples_per_second': 170.636, 'eval_steps_per_second': 10.816, 'epoch': 10.0}


## Guardando Modelo y Tokenizer

In [27]:
# os.mkdir("fge-qa-model")
model.save_pretrained(".fge-qa-model/fine-tuned-qa-model")
tokenizer.save_pretrained(".fge-qa-model/fine-tuned-qa-model")

('.fge-qa-model/fine-tuned-qa-model/tokenizer_config.json',
 '.fge-qa-model/fine-tuned-qa-model/special_tokens_map.json',
 '.fge-qa-model/fine-tuned-qa-model/vocab.txt',
 '.fge-qa-model/fine-tuned-qa-model/added_tokens.json',
 '.fge-qa-model/fine-tuned-qa-model/tokenizer.json')

In [28]:
trainer.push_to_hub()

model.safetensors:   0%|          | 0.00/437M [00:00<?, ?B/s]

events.out.tfevents.1738687548.447c5f9076b1.2059.0:   0%|          | 0.00/5.09k [00:00<?, ?B/s]

events.out.tfevents.1738687710.447c5f9076b1.2059.1:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

Upload 5 LFS files:   0%|          | 0/5 [00:00<?, ?it/s]

events.out.tfevents.1738689171.447c5f9076b1.2059.2:   0%|          | 0.00/359 [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.30k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/LeninGF/fge-robos-qa-model/commit/85ac35e38ebda4b2f63abb3bfbdb6d73e5e2adca', commit_message='End of training', commit_description='', oid='85ac35e38ebda4b2f63abb3bfbdb6d73e5e2adca', pr_url=None, repo_url=RepoUrl('https://huggingface.co/LeninGF/fge-robos-qa-model', endpoint='https://huggingface.co', repo_type='model', repo_id='LeninGF/fge-robos-qa-model'), pr_revision=None, pr_num=None)

## Demo

In [40]:
import numpy as np
from pprint import pprint
preguntas_comunes = [
    "¿Qué objetos fueron robados?",
    "¿En qué fecha ocurrió el incidente?",
    "¿A qué hora sucedió el robo?",
    "¿En qué dirección o entre qué calles sucedió el robo, sucedo o incidente?",
    "¿Qué valor en dólares tenían los objetos sustraídos o robados?",
    # "¿Existió intimidación, agresión o violencia?"
]
# context = "es el caso señor fiscal que el dia de hoy 28 de julio del 2016 siendo aproximadamente las 17h00 en circunstancias que me baje de un bus en la parroquia san camilo con la finalidad de dirigirme a mi lugar de trabajo esto el taller eco frio de repente al llegar a la altura del cuerpo de bomberos fui inteceptado por dos sujetos inidentificados que se movilizaban a bordo de una motocicleta marca suzuki colo rojo sin placas los mismos que con un arma de fuego me intimidaron acto seguido procedieron a robarme ciento ochenta dolares en efectivo dinero que era de producto de mi trabajo luego se dieron a la fuga con rumbo desconocido por tal motivo solicito se realicen las respectivas investigaciones"
random_idx = np.random.randint(0, squad["test"].num_rows)
context = squad["test"][random_idx]["context"]
print(f"Test sample: {random_idx}")
pprint(context)

Test sample: 435
('el día de ayer 17 de junio del 2020 aproximadamente a las 10h00 estaba '
 'realizando una factura en el sector de cotocollao av legarda instantes en '
 'los cuales fui sorprendido por un sujeto de acento costeño mismos que bajo '
 'amenazas y con una cuchillo procedido a sustraerme lo siguiente un celular '
 'marca samsun modelo a10 s duo color negro de la operadora movistar asignado '
 'con el número 0995781523 imei 358099 10 7234734 perteneciente a la empresa '
 'alimentos yupi con lo antes expuesto solicito las respectivas '
 'investigaciones para mayor información comunicarse a los números 0983771221')


In [41]:
from transformers import pipeline

question_answerer = pipeline("question-answering", model="LeninGF/fge-robos-qa-model")
for question in preguntas_comunes:
    answer = question_answerer(question=question, context=context)
    pprint(question)
    pprint(answer)

Device set to use cuda:0


'¿Qué objetos fueron robados?'
{'answer': 'un celular marca samsun modelo a10 s duo color negro',
 'end': 336,
 'score': 0.9996917247772217,
 'start': 284}
'¿En qué fecha ocurrió el incidente?'
{'answer': 'el día de ayer 17 de junio del 2020',
 'end': 35,
 'score': 0.9960949420928955,
 'start': 0}
'¿A qué hora sucedió el robo?'
{'answer': 'aproximadamente a las 10h00',
 'end': 63,
 'score': 0.9906905889511108,
 'start': 36}
'¿En qué dirección o entre qué calles sucedió el robo, sucedo o incidente?'
{'answer': 'av legarda', 'end': 131, 'score': 0.6637033224105835, 'start': 121}
'¿Qué valor en dólares tenían los objetos sustraídos o robados?'
{'answer': 'un celular marca samsun modelo a10 s duo color negro',
 'end': 336,
 'score': 0.008780337870121002,
 'start': 284}


# TODO

- Evaluar el modelo sin hacer fine tuning
- Generar un branch para cambios del programa
- Incluir otras metricas de evaluacion para el entrenamiento
